# 04 · Reverse shortest paths and ignition origin scoring

For every candidate cell, optimise a latent ignition time and compare arrival upper bounds to each detected hotspot. Revisit gaps motivate a weak early-arrival penalty.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import json
import matplotlib.pyplot as plt
INPUT = ROOT / "data/synthetic/toy_firms.csv"  # SYNTHETIC DEMO, NOT historical NASA data
WEATHER = ROOT / "data/synthetic/toy_weather.json"
assert INPUT.exists(), "Run `python scripts/make_synthetic.py` from repository root first"
print("Data mode: SYNTHETIC - demonstration only")

In [ ]:
from wildfire.ingest import load_firms,load_weather
from wildfire.inference import infer_origin,InferenceOptions,public_result
sat=load_firms(INPUT)
wx=load_weather(WEATHER)
options=InferenceOptions(cell_m=850,margin_km=9,ensemble=True)
result=infer_origin(sat,wx,options,variant='fusion')
print('Status:',result['status'],'Best:',result['best_origin'])
print('Top-80% model-score area:',result['regions']['top80']['area_km2'],'km²')
for warning in result['warnings']:print('CAVEAT:',warning)

In [ ]:
# False-colour scores visualize a rank field, not calibrated posterior confidence.
fig,ax=plt.subplots(figsize=(10,6))
candidates=result['candidate_cells']
values=[c['score_mass'] for c in candidates]
sc=ax.scatter([c['longitude'] for c in candidates],[c['latitude'] for c in candidates],
              c=values,cmap='magma',s=22)
ax.scatter(result['best_origin']['longitude'],result['best_origin']['latitude'],
           marker='*',s=180,c='cyan',edgecolors='black',label='Top candidate')
ax.scatter(result['baseline_centroid']['longitude'],result['baseline_centroid']['latitude'],
           marker='x',s=125,c='lime',label='Earliest centroid')
ax.set(xlabel='Longitude',ylabel='Latitude',title='Synthetic model-relative origin ranking')
ax.legend();plt.colorbar(sc,ax=ax,label='Normalised score mass per grid cell');plt.show()

In [ ]:
out=ROOT/'artifacts/04_inference.json'
out.write_text(json.dumps(public_result(result),indent=2))
print('Saved model output:',out)
print('Evaluation label has not been loaded in this notebook.')

**Interpretation:** top-80%/90% score regions have NOT been empirically calibrated; this is a weighted candidate ranking, not a forensic probability or proof of utility involvement.